# Chapter 7 — The Jeans equations: weighing galaxies with velocity dispersions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/07_jeans_equations.ipynb)

**Goal.** We rarely know the orbits of individual stars, but we can measure averages: how many stars
there are at each radius, and how fast they move on average. The Jeans equations connect these averages
to the gravitational potential, and so to the mass, including the dark matter.

**You will learn**
1. the **velocity moments** of a distribution function: density, mean velocity, velocity dispersion, anisotropy $\beta$;
2. the **spherical Jeans equation**, how to solve it, and how to check it against AGAMA;
3. to **measure a mass profile** from the kinematics of a star sample;
4. what observers actually see (**line-of-sight dispersions**), and the **mass–anisotropy degeneracy**;
5. the **asymmetric drift** of disk stars.

**Units.** $M_\odot$, kpc, km/s, as in Chapters 4–6.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object
G = agama.G                                        # 4.3e-6 kpc (km/s)^2 / Msun

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import minimize

rng = np.random.default_rng(7)

## 7.1 Velocity moments and the spherical Jeans equation

**Moments.** Given the distribution function $f(\mathbf x, \mathbf v)$ of a population of stars (Chapter 6),
the quantities an observer can hope to measure are its velocity moments:

$$ \nu(\mathbf x) = \int f\, d^3v, \qquad \bar v_i = \frac{1}{\nu}\int v_i f\, d^3v, \qquad
   \sigma_{ij}^2 = \frac{1}{\nu}\int (v_i - \bar v_i)(v_j - \bar v_j) f\, d^3v . $$

$\nu$ is the number density of the stars (we call it $\nu$, not $\rho$, because the stars we observe need
not be what produces the gravity: they are **tracers**). In a spherical, non-rotating system the
dispersion has a radial part $\sigma_r$ and two equal tangential parts $\sigma_\theta = \sigma_\phi$. Their
ratio defines the **velocity anisotropy**

$$ \beta(r) = 1 - \frac{\sigma_\theta^2 + \sigma_\phi^2}{2\sigma_r^2}: \qquad
   \beta = 0 \text{ isotropic}, \quad 0 < \beta \le 1 \text{ radially biased}, \quad \beta < 0 \text{ tangentially biased}. $$

**The Jeans equation.** Multiply the collisionless Boltzmann equation by $v_r$ and integrate over all
velocities (Binney & Tremaine §4.8). For a steady spherical system the result is

$$ \frac{d(\nu\sigma_r^2)}{dr} + \frac{2\beta}{r}\,\nu\sigma_r^2 = -\nu\,\frac{d\Phi}{dr} = -\nu\,\frac{G M(r)}{r^2} . $$

It is a hydrostatic-equilibrium equation: the "pressure" $\nu\sigma_r^2$ of the stars balances gravity.
Given $\nu(r)$, $\beta(r)$ and $M(r)$, it is a first-order equation for $\nu\sigma_r^2$. With the integrating
factor $g(r) = \exp\!\int^r 2\beta(r')/r'\,dr'$ (for constant $\beta$, $g = r^{2\beta}$) and the boundary condition
$\nu\sigma_r^2 \to 0$ as $r \to \infty$:

$$ \nu\sigma_r^2(r) = \frac{1}{g(r)}\int_r^\infty g(r')\,\nu(r')\,\frac{G M(r')}{r'^2}\,dr' . $$

We evaluate both integrals on a logarithmic grid of radii with the trapezoidal rule.

In [ ]:
r_grid = np.geomspace(1e-4, 1e3, 4000)               # kpc; log spacing resolves both small and large r

def jeans_nusr2(nu, M, beta):
    """nu * sigma_r^2 on r_grid from the spherical Jeans equation.
    nu, M, beta: functions of r (tracer density, enclosed mass, anisotropy)."""
    g = np.exp(cumulative_trapezoid(2 * beta(r_grid) / r_grid, r_grid, initial=0))   # integrating factor
    integrand = g * nu(r_grid) * G * M(r_grid) / r_grid**2
    # integral from r to infinity = (integral from r to the last grid point): cumulate from the outer end
    outer = -cumulative_trapezoid(integrand[::-1], r_grid[::-1], initial=0)[::-1]
    return outer / g

# Test on the Plummer sphere (Chapter 1): isotropic, self-consistent (tracer = mass), known answer
Mp, b = 1e10, 1.0
nu_plummer = lambda r: 3 * Mp / (4 * np.pi * b**3) * (1 + r**2 / b**2)**-2.5   # mass density; any normalisation works
M_plummer = lambda r: Mp * r**3 / (r**2 + b**2)**1.5
sr2_jeans = jeans_nusr2(nu_plummer, M_plummer, lambda r: 0 * r) / nu_plummer(r_grid)
sr2_exact = G * Mp / (6 * np.sqrt(r_grid**2 + b**2))

# The same with AGAMA: build the isotropic DF of the Plummer sphere and compute its moments directly
pot_p = agama.Potential(type='Plummer', mass=Mp, scaleRadius=b)
df_p = agama.DistributionFunction(type='QuasiSpherical', potential=pot_p, density=pot_p)  # f(E) by Eddington inversion (Ch. 8)
gm_p = agama.GalaxyModel(pot_p, df_p)                 # a DF living in a potential: can compute moments and sample
r_test = np.array([0.1, 0.5, 1, 2, 5, 10])
dens, sig2 = gm_p.moments(np.column_stack([r_test, 0 * r_test, 0 * r_test]), dens=True, vel=False, vel2=True)
# sig2 has columns xx, yy, zz, xy, xz, yz; at points on the x-axis xx is the radial component
sr2_agama = sig2[:, 0]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(r_grid, np.sqrt(sr2_exact), 'k-', lw=3, alpha=0.3, label='exact $GM/6\\sqrt{r^2+b^2}$')
ax.loglog(r_grid, np.sqrt(sr2_jeans), 'C0--', label='Jeans equation (numerical)')
ax.loglog(r_test, np.sqrt(sr2_agama), 'C1o', label='AGAMA GalaxyModel.moments')
ax.set_xlim(1e-2, 1e2); ax.set_xlabel('$r$ [kpc]'); ax.set_ylabel('$\\sigma_r$ [km/s]'); ax.legend()
plt.show()

inside = r_grid < 100                                 # away from the truncated outer end of the grid
err_jeans = np.max(np.abs(sr2_jeans[inside] / sr2_exact[inside] - 1))
err_agama = np.max(np.abs(sr2_agama / (G * Mp / (6 * np.sqrt(r_test**2 + b**2))) - 1))
print(f'max relative error of sigma_r^2: Jeans {err_jeans:.1e}, AGAMA {err_agama:.1e}')
assert err_jeans < 1e-4 and err_agama < 1e-4

Both agree with the analytic result to better than $10^{-4}$. AGAMA's `moments` integrates the DF over
velocities numerically, so it needs no Jeans equation at all: it is an independent check.

## 7.2 Weighing a dwarf galaxy with 3-d kinematics

Now a case where the tracer is **not** the mass. Dwarf spheroidal galaxies are dominated by dark matter.
Our model:

- **dark halo**: NFW (Chapter 4), $\rho \propto 1/[x(1+x)^2]$ with $x = r/r_s$, $r_s = 1$ kpc, and
  $M(r) = M_{\rm n}\,[\ln(1+x) - x/(1+x)]$ with $M_{\rm n} = 10^9\,M_\odot$ (AGAMA's `mass` parameter for NFW);
- **stars**: a Plummer profile with scale radius $a = 0.25$ kpc; their own mass is neglected;
- **anisotropy**: Osipkov–Merritt, $\beta(r) = r^2/(r^2 + r_a^2)$ with $r_a = 1$ kpc: isotropic in the centre, radial outside.

AGAMA builds the corresponding DF and draws stars from it. We then pretend to be observers with
perfect 3-d positions and velocities of $2\times10^5$ stars. In radial bins we measure $\nu$, $\sigma_r^2$ and
$\beta$, and solve the Jeans equation for the mass:

$$ M(r) = -\frac{r\,\sigma_r^2}{G}\left[\frac{d\ln\nu}{d\ln r} + \frac{d\ln\sigma_r^2}{d\ln r} + 2\beta\right]. $$

In [ ]:
a, Mn, rs = 0.25, 1e9, 1.0
halo = agama.Potential(type='NFW', mass=Mn, scaleRadius=rs)
stars = agama.Density(type='Plummer', mass=1, scaleRadius=a)     # tracer density; mass 1 = fraction of stars
nu_star = lambda r: 3 / (4 * np.pi * a**3) * (1 + r**2 / a**2)**-2.5
def M_nfw(r, Mn=Mn, rs=rs):
    x = r / rs
    return Mn * (np.log1p(x) - x / (1 + x))
print(f'check M_nfw against AGAMA at 1 kpc: {M_nfw(1.0):.6e} vs {halo.enclosedMass(1.0):.6e}')
assert abs(M_nfw(1.0) / halo.enclosedMass(1.0) - 1) < 1e-8

r_a = 1.0
df = agama.DistributionFunction(type='QuasiSpherical', potential=halo, density=stars, r_a=r_a)
xv, _ = agama.GalaxyModel(halo, df).sample(200000)    # returns positions+velocities (N, 6) and masses
r = np.linalg.norm(xv[:, :3], axis=1)
v_r = np.sum(xv[:, :3] * xv[:, 3:], axis=1) / r        # radial velocity x.v / r
v2 = np.sum(xv[:, 3:]**2, axis=1)

edges = np.geomspace(0.04, 2.0, 15)
rc = np.sqrt(edges[1:] * edges[:-1])                  # geometric bin centres
n = np.histogram(r, edges)[0]                         # stars per shell
nu_m = n / (4 / 3 * np.pi * np.diff(edges**3)) / len(r)
sr2_m = np.histogram(r, edges, weights=v_r**2)[0] / n # mean of v_r^2 (the mean v_r is zero)
st2_m = (np.histogram(r, edges, weights=v2)[0] / n - sr2_m) / 2   # each tangential component
beta_m = 1 - st2_m / sr2_m

dlog = lambda y: np.gradient(np.log(y), np.log(rc))   # d ln y / d ln r by finite differences
M_jeans = -rc * sr2_m / G * (dlog(nu_m) + dlog(sr2_m) + 2 * beta_m)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(rc, beta_m, 'o', label='measured from the sample')
ax[0].plot(rc, rc**2 / (rc**2 + r_a**2), 'k-', label='Osipkov–Merritt $r^2/(r^2+r_a^2)$')
ax[0].set_xscale('log'); ax[0].set_xlabel('$r$ [kpc]'); ax[0].set_ylabel('$\\beta$'); ax[0].legend()
ax[1].loglog(rc, M_jeans, 'o', label='Jeans equation on $2\\times10^5$ stars')
ax[1].loglog(rc, M_nfw(rc), 'k-', label='true NFW $M(r)$')
ax[1].set_xlabel('$r$ [kpc]'); ax[1].set_ylabel('$M(<r)$ [$M_\\odot$]'); ax[1].legend()
plt.tight_layout(); plt.show()

ratio = M_jeans / M_nfw(rc)
print('M_Jeans / M_true per bin:', np.round(ratio, 3))
interior = slice(2, -1)                               # first bins: few stars; last bin: one-sided derivative
print(f'median |error| in the interior bins: {np.median(np.abs(ratio[interior] - 1)):.3f}')
assert np.median(np.abs(ratio[interior] - 1)) < 0.05

With full 3-d information the Jeans equation recovers the dark-matter mass profile to a few per cent.
The innermost bin is poor: few stars there, and a derivative of noisy data. The last bin uses a
one-sided difference.

## 7.3 What observers see: line-of-sight dispersions

For a distant galaxy we know only projected positions $R$ and line-of-sight velocities. Two profiles
are measurable: the surface density $\Sigma(R)$ and the line-of-sight dispersion $\sigma_{\rm los}(R)$. The
line of sight at projected radius $R$ meets the star at radius $r$ at an angle; its velocity along it is a
mix of $v_r$ and $v_\theta$, and the integral over the line of sight gives (Binney & Mamon 1982)

$$ \Sigma\,\sigma_{\rm los}^2(R) = 2\int_R^\infty \left(1 - \beta\frac{R^2}{r^2}\right)\frac{\nu\sigma_r^2\, r\,dr}{\sqrt{r^2 - R^2}},
   \qquad \Sigma(R) = 2\int_R^\infty \frac{\nu\, r\,dr}{\sqrt{r^2 - R^2}} . $$

The $1/\sqrt{r^2 - R^2}$ is infinite at $r = R$; the substitution $r = \sqrt{R^2 + u^2}$ removes it
($r\,dr/\sqrt{r^2-R^2} = du$), leaving smooth integrals over $u$ from 0 to $\infty$.

In [ ]:
u_grid = np.geomspace(1e-5, 1e3, 3000)

def sigma_los2(R, M, beta, nu=nu_star):
    """Line-of-sight velocity dispersion squared at projected radii R (array) from the Jeans equation."""
    p = jeans_nusr2(nu, M, beta)
    out = []
    for Ri in R:
        rr = np.sqrt(Ri**2 + u_grid**2)
        p_rr = np.exp(np.interp(np.log(rr), np.log(r_grid), np.log(p)))   # interpolate in log-log
        out.append(np.trapezoid((1 - beta(rr) * Ri**2 / rr**2) * p_rr, u_grid) / np.trapezoid(nu(rr), u_grid))
    return np.array(out)

# four anisotropy models for the same stars in the same halo
models = {
    'tangential, $\\beta = -0.5$': (dict(beta0=-0.5), lambda r: -0.5 + 0 * r),
    'isotropic, $\\beta = 0$':     (dict(),           lambda r: 0 * r),
    'OM, $r_a = 1$ kpc':           (dict(r_a=1.0),    lambda r: r**2 / (r**2 + 1.0**2)),
    'OM, $r_a = 0.5$ kpc':         (dict(r_a=0.5),    lambda r: r**2 / (r**2 + 0.5**2)),
}
R_obs = np.geomspace(0.03, 1.5, 20)
fig, ax = plt.subplots(figsize=(7, 4.5))
for j, (name, (kw, beta)) in enumerate(models.items()):
    s2 = sigma_los2(R_obs, M_nfw, beta)
    # AGAMA: points with 2 coordinates (X, Y) ask for moments projected along Z
    df_k = agama.DistributionFunction(type='QuasiSpherical', potential=halo, density=stars, **kw)
    _, m2 = agama.GalaxyModel(halo, df_k).moments(np.column_stack([R_obs, 0 * R_obs]), dens=True, vel=False, vel2=True)
    print(f'{name:28s} max |Jeans/AGAMA - 1| = {np.max(np.abs(s2 / m2[:, 2] - 1)):.1e}')
    assert np.max(np.abs(s2 / m2[:, 2] - 1)) < 2e-2
    ax.semilogx(R_obs, np.sqrt(s2), '-', color=f'C{j}', label=name)
    ax.semilogx(R_obs[::3], np.sqrt(m2[::3, 2]), 'o', color=f'C{j}', ms=4)
ax.set_xlabel('$R$ [kpc]'); ax.set_ylabel('$\\sigma_{\\rm los}$ [km/s]')
ax.set_title('same halo, same stars, different $\\beta$ (lines: Jeans; dots: AGAMA)')
ax.legend(fontsize=9); plt.show()

Our projected Jeans solution agrees with AGAMA's projected DF moments to better than 0.2% for the first
three models. For $r_a = 0.5$ kpc it is 1%: strongly radial models are close to the limit where the DF
becomes negative, and AGAMA's DF is then slightly approximate.

The four profiles differ: radial orbits make $\sigma_{\rm los}$ high in the centre (we look along their
radial motion) and low outside. Does that mean we can measure $\beta$ from $\sigma_{\rm los}(R)$? Only if we
already know the mass profile.

## 7.4 The mass–anisotropy degeneracy

Treat the isotropic model as the data. An observer who does not know $\beta$ assumes one of the other
anisotropy profiles and fits the halo, i.e. its two parameters $(M_{\rm n}, r_s)$, to $\sigma_{\rm los}^2(R)$.
We minimise $\sum_i [\sigma^2_{\rm model}(R_i)/\sigma^2_{\rm data}(R_i) - 1]^2$ with the Nelder–Mead simplex method
(`scipy.optimize.minimize`), fitting $\log_{10}$ of the parameters so they stay positive.

In [ ]:
data = sigma_los2(R_obs, M_nfw, models['isotropic, $\\beta = 0$'][1])
r_half = a / np.sqrt(2**(2 / 3) - 1)                  # 3-d radius containing half of the Plummer stars
print(f'3-d half-light radius r_1/2 = {r_half:.3f} kpc (projected: a = {a} kpc)')

r_plot = np.geomspace(0.03, 3, 100)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].semilogx(R_obs, np.sqrt(data), 'ko', label='"data" (isotropic)')
fits = {}
for j, (name, (kw, beta)) in enumerate(models.items()):
    def misfit(logp):
        return np.sum((sigma_los2(R_obs, lambda r: M_nfw(r, *10**logp), beta) / data - 1)**2)
    res = minimize(misfit, x0=[9.0, 0.0], method='Nelder-Mead', options=dict(xatol=1e-4, fatol=1e-12))
    Mn_fit, rs_fit = 10**res.x
    fits[name] = (Mn_fit, rs_fit)
    s_fit = np.sqrt(sigma_los2(R_obs, lambda r: M_nfw(r, Mn_fit, rs_fit), beta))
    rms = np.sqrt(np.mean((s_fit / np.sqrt(data) - 1)**2))
    ax[0].semilogx(R_obs, s_fit, '-', color=f'C{j}', label=f'{name}: rms {rms:.1%}')
    ax[1].semilogx(r_plot, M_nfw(r_plot, Mn_fit, rs_fit) / M_nfw(r_plot), '-', color=f'C{j}', label=name)
    print(f'{name:28s} M_n = {Mn_fit:.2e}, r_s = {rs_fit:6.2f} kpc, rms misfit of sigma_los {rms:.1%}, '
          f'M/M_true at 0.3, 1, 3 r_1/2: ' + ', '.join(f'{M_nfw(x * r_half, Mn_fit, rs_fit) / M_nfw(x * r_half):.2f}' for x in (0.3, 1, 3)))
ax[0].set_xlabel('$R$ [kpc]'); ax[0].set_ylabel('$\\sigma_{\\rm los}$ [km/s]'); ax[0].legend(fontsize=8)
ax[1].axvline(r_half, color='gray', ls=':'); ax[1].text(r_half * 1.05, 1.45, '$r_{1/2}$', color='gray')
ax[1].set_xlabel('$r$ [kpc]'); ax[1].set_ylabel('fitted $M(r)$ / true $M(r)$'); ax[1].set_ylim(0.4, 1.6)
plt.tight_layout(); plt.show()

# where do the four fitted mass profiles agree best?  spread = max - min of M_fit/M_true across the fits
spread = np.ptp([M_nfw(r_plot, *p) / M_nfw(r_plot) for p in fits.values()], axis=0)
r_pinch = r_plot[np.argmin(spread)]
print(f'fitted masses agree best at r = {r_pinch:.2f} kpc = {r_pinch / r_half:.1f} r_1/2 (spread {spread.min():.2f}); '
      f'spread at r_1/2: {np.interp(r_half, r_plot, spread):.2f}, at 0.1 kpc: {np.interp(0.1, r_plot, spread):.2f}')
assert spread.min() < 0.1 and np.interp(0.1, r_plot, spread) > 0.4

Every assumed $\beta$ fits the data with an rms misfit of 2–6% in $\sigma_{\rm los}$. With $N$ stars in a radial bin,
$\sigma$ is measured to about $1/\sqrt{2N}$ — 7% for 100 stars — so typical data could barely tell these models
apart. But they disagree about the mass: from $-35\%$ to $+45\%$ in the centre, and by similar amounts
at large radii. This is the **mass–anisotropy degeneracy**: $\sigma_{\rm los}$ cannot separate "more mass" from
"more radial orbits". (Here we only allowed NFW halos; with freer mass profiles it is worse.)

The fitted mass profiles cross at one radius, here about $0.6$ kpc. Where exactly they pinch depends on the
family of halos we allowed and on how far out the data go. Is there a mass that is robust to $\beta$ for a
more fundamental reason? Wolf et al. (2010) proposed the estimator

$$ M(r_{1/2}) \approx \frac{3\,\langle\sigma_{\rm los}^2\rangle\, r_{1/2}}{G}, $$

where $\langle\sigma_{\rm los}^2\rangle$ is weighted by the surface density and averaged over the whole galaxy.
A remarkable fact makes this work: $\langle\sigma_{\rm los}^2\rangle$ does not depend on $\beta$ **at all**. It equals
$\tfrac13\langle v^2\rangle$, and the virial theorem for tracers (Chapter 1 §1.4) fixes $\langle v^2\rangle = \langle r\, d\Phi/dr\rangle$
regardless of how the kinetic energy is shared between radial and tangential motion.

In [ ]:
R_all = np.geomspace(1e-3, 30, 400)
Sigma = (1 + R_all**2 / a**2)**-2 / (np.pi * a**2)    # Plummer surface density (normalised to 1)
for name, (kw, beta) in models.items():
    s2_mean = np.trapezoid(Sigma * sigma_los2(R_all, M_nfw, beta) * R_all, R_all) / np.trapezoid(Sigma * R_all, R_all)
    print(f'{name:28s} <sigma_los^2>^(1/2) = {np.sqrt(s2_mean):.4f} km/s,  '
          f'Wolf M(r_1/2) / true = {3 * s2_mean * r_half / G / M_nfw(r_half):.3f}')
assert abs(3 * s2_mean * r_half / G / M_nfw(r_half) - 1) < 0.1

# check the virial identity <v^2> = <r dPhi/dr> = <G M(r)/r> with the stars sampled in 7.2
print(f'from the 7.2 sample: <v^2>/3 = {np.mean(v2) / 3:.2f}, <G M(r)/r>/3 = {np.mean(G * M_nfw(r) / r) / 3:.2f} (km/s)^2')
assert abs(np.mean(v2) / np.mean(G * M_nfw(r) / r) - 1) < 0.01

All four anisotropies give the same $\langle\sigma_{\rm los}^2\rangle$ (to $10^{-5}$), so the estimator gives the same mass,
4% above the true $M(r_{1/2})$, whatever $\beta$ is. Its accuracy, unlike its $\beta$-independence, is not exact:
it rests on an argument about the mass profile near $r_{1/2}$ and has to be tested model by model (Exercise 3).
This is why masses of dwarf galaxies are usually quoted within the half-light radius.

## 7.5 Asymmetric drift in the Milky Way disk

The Jeans equations also apply to disks. In the plane of an axisymmetric disk, the radial Jeans equation
(Binney & Tremaine eq. 4.228) gives the **asymmetric drift**: a population with random motions rotates
more slowly than the circular speed,

$$ v_c^2 - \bar v_\phi^2 = \sigma_R^2\left[\frac{\sigma_\phi^2}{\sigma_R^2} - 1 - \frac{\partial\ln\nu}{\partial\ln R}
   - \frac{\partial\ln\sigma_R^2}{\partial\ln R} - \frac{R}{\sigma_R^2}\frac{\partial\,\overline{v_R v_z}}{\partial z}\right]. $$

The hotter the population, the larger the lag: its pressure supports part of its weight, so it needs
less rotation. Dehnen & Binney (1998) found this in Hipparcos data: the mean rotation of stars near the
Sun falls linearly with $\sigma^2$.

We test it with AGAMA's `QuasiIsothermal` DF for disk stars (Chapter 10 explains it) in MWPotential2014:
exponential disks with scale length 2.5 kpc and scale height 0.3 kpc, with three different radial
dispersions. `moments` gives $\nu$, $\bar v_\phi$ and the dispersions; the derivatives are finite differences.

In [ ]:
mw = agama.Potential(os.path.join(os.path.dirname(agama.__file__), 'data', 'MWPotential2014.ini'))
R0, d = 8.0, 0.1
vc2 = -R0 * mw.force([R0, 0, 0])[0]
pts = np.array([[R0 - d, 0, 0], [R0, 0, 0], [R0 + d, 0, 0], [R0, 0, d], [R0, 0, -d]])   # at y = 0: phi-direction = y
lag, sig2R = [], []
for sigR_8 in [20.0, 35.0, 50.0]:                    # approximate sigma_R at 8 kpc
    df_d = agama.DistributionFunction(type='QuasiIsothermal', potential=mw, mass=1, Rdisk=2.5, Hdisk=0.3,
                                      sigmar0=sigR_8 * np.exp(R0 / 5.0), Rsigmar=5.0)   # sigma_R ~ sigmar0 exp(-R/Rsigmar)
    nu_d, vbar, v2m = agama.GalaxyModel(mw, df_d).moments(pts, dens=True, vel=True, vel2=True)
    # v2m columns: xx, yy, zz, xy, xz, yz; second moments about the mean except where noted
    vphi = vbar[1, 1]
    sR2 = v2m[:, 0]                                  # sigma_R^2 at each point (v_R = v_x on the x-axis)
    sphi2 = v2m[1, 1] - vphi**2                      # <v_phi^2> - vbar_phi^2  (the yy moment includes the mean)
    dlnnu = (np.log(nu_d[2]) - np.log(nu_d[0])) / (2 * d) * R0
    dlnsR2 = (np.log(sR2[2]) - np.log(sR2[0])) / (2 * d) * R0
    dvRvz = (v2m[3, 4] - v2m[4, 4]) / (2 * d)        # xz moment above minus below the plane
    pred = np.sqrt(vc2 - sR2[1] * (sphi2 / sR2[1] - 1 - dlnnu - dlnsR2 - R0 * dvRvz / sR2[1]))
    print(f'sigma_R = {np.sqrt(sR2[1]):5.1f} km/s: vbar_phi from the DF {vphi:6.2f}, from the Jeans equation {pred:6.2f} km/s '
          f'(v_c = {np.sqrt(vc2):.2f})')
    assert abs(pred - vphi) < 1.0
    lag.append(np.sqrt(vc2) - vphi); sig2R.append(sR2[1])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sig2R, lag, 'o-')
ax.set_xlabel('$\\sigma_R^2$ [(km/s)$^2$]'); ax.set_ylabel('$v_c - \\bar v_\\phi$ [km/s]')
ax.set_title('asymmetric drift at $R$ = 8 kpc'); ax.set_xlim(0); ax.set_ylim(0)
plt.show()

The Jeans equation reproduces the mean rotation computed directly from the DF to better than 1 km/s
(the residual comes from the finite differences and AGAMA's numerical integration). The lag grows
from 6 to 33 km/s, roughly in proportion to $\sigma_R^2$.

## Summary

- The Jeans equations are velocity moments of the collisionless Boltzmann equation. They relate the
  density and velocity dispersions of a tracer to the potential, without knowing the full DF.
- Spherical Jeans: $d(\nu\sigma_r^2)/dr + 2\beta\nu\sigma_r^2/r = -\nu GM/r^2$. With 3-d kinematics it gives $M(r)$ directly.
- With line-of-sight velocities only, mass and anisotropy are degenerate. The mass inside the
  half-light radius is robust: $M(r_{1/2}) \approx 3\langle\sigma_{\rm los}^2\rangle r_{1/2}/G$.
- In disks, the radial Jeans equation gives the asymmetric drift: hotter populations rotate more slowly.

| task | AGAMA |
|---|---|
| DF of a spherical tracer in a given potential | `agama.DistributionFunction(type='QuasiSpherical', potential=, density=, beta0= or r_a=)` |
| moments of a DF | `agama.GalaxyModel(pot, df).moments(points, dens=, vel=, vel2=)`; $(N, 2)$ points give projected moments |
| mock star sample | `xv, m = agama.GalaxyModel(pot, df).sample(N)` |
| disk DF | `agama.DistributionFunction(type='QuasiIsothermal', potential=, Rdisk=, Hdisk=, sigmar0=, Rsigmar=)` |

## Exercises

1. **Plummer with anisotropy.** Solve the Jeans equation for the self-consistent Plummer sphere with $\beta = -0.3$
   and compare $\sigma_r$ with AGAMA `moments` for `beta0=-0.3`.
2. **Fewer stars.** Repeat §7.2 with 2000 stars (a realistic number for a dwarf galaxy, though with 3-d velocities).
   How large are the errors on $M(r)$? Use bootstrap resampling to estimate them.
3. **Cusp or core?** Replace the NFW halo by a cored halo (e.g. `type='Spheroid'` with `gamma=0`) of similar
   $M(r_{1/2})$, and compare $\sigma_{\rm los}(R)$ for isotropic stars with the NFW + OM models of §7.4. Can a core be told from radial anisotropy?
   How accurate is the Wolf estimator for the cored halo?
4. **The cusp–anisotropy theorem.** Try `beta0=0.5` for the Plummer stars in §7.3. AGAMA returns nonsense:
   a cored tracer cannot have radially biased orbits in its centre (An & Evans 2006 showed
   $\gamma \ge 2\beta$ at the centre, where $\nu \propto r^{-\gamma}$).
5. **Asymmetric drift slope.** Add two more values of $\sigma_R$ in §7.5 and fit $v_c - \bar v_\phi = \sigma_R^2/k$. Compare $k$
   with the value Dehnen & Binney (1998) measured.

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §4.8.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapters 5 and 6: https://galaxiesbook.org
- J. Binney & G. A. Mamon (1982), MNRAS 200, 361: https://ui.adsabs.harvard.edu/abs/1982MNRAS.200..361B
- J. Wolf et al. (2010), "Accurate masses for dispersion-supported galaxies", MNRAS 406, 1220:
  https://ui.adsabs.harvard.edu/abs/2010MNRAS.406.1220W
- J. H. An & N. W. Evans (2006), "A cusp slope–central anisotropy theorem", ApJ 642, 752:
  https://ui.adsabs.harvard.edu/abs/2006ApJ...642..752A
- W. Dehnen & J. Binney (1998), "Local stellar kinematics from Hipparcos data", MNRAS 298, 387:
  https://ui.adsabs.harvard.edu/abs/1998MNRAS.298..387D